# Sorani (Kurdish) Voice Training - Piper TTS (Seçenek A)

Bu not defteri, kadın Sorani (ckb) sesini Piper TTS'e eğitir. Veri kaynağı: Common Voice ckb (HuggingFace) üzerinden. Eğitim sonunda model.onnx + model.onnx.json dosyaları üretilir.

Adimlar: 1) kurulum  2) veri indir (kadın konuşmacı seç)  3) veriyi hazırla  4) eğit  5) ONNX aktar + indir

In [ ]:
# 0) GPU kontrol
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "GPU yok"

In [ ]:
!pip install -q piper-training datasets
print("kuruldu")

## 2) Veri kaynağını seç

**Aşağıdaki ilk hücreyi çalıştırarak hangi veriyle eğiteceğini seç.**
- `commonvoice` → Common Voice ckb kadın konuşmacı (Seçenek A)
- `synthetic` → kurdishtts ile üretilmiş sentetik Sorani (synthetic.zip yükle)
- `recording` → Recording Studio kayıtları (`wav+txt` zip yükle)

Eğer `synthetic` veya `recording` seçtiysen Common Voice hücresini ATLA ve doğrudan ilgili yükleme hücresini çalıştır.

In [ ]:
# 2) VERİ KAYNAĞI
# "commonvoice" | "synthetic" | "recording"
DATA_SOURCE = "synthetic"
TARGET_HOURS = 1.0
print("Kaynak:", DATA_SOURCE)

### (A) Common Voice — otomatik kadın konuşmacı
Bu hücreyi sadece `DATA_SOURCE == "commonvoice"` ise çalıştır.

In [ ]:
# 2.A) Common Voice ckb indir (kadın konuşmacı) — SEÇENEK A
import os, glob, librosa, soundfile as sf
import numpy as np
from datasets import load_dataset

os.makedirs("data", exist_ok=True)
try:
    ds = load_dataset("mozilla-foundation/common_voice_17_0", "ckb", split="train", streaming=False)
except Exception as e:
    print("17_0 denendi:", e)
    ds = load_dataset("mozilla-foundation/common_voice_13_0", "ckb", split="train")

df = ds.to_pandas()
f = df[df["gender"] == "female"]
print("kadın kayıt:", len(f))
top = f["client_id"].value_counts().idxmax()
sub = f[f["client_id"] == top]
print("seçilen konuşmacı:", top, "kayıt:", len(sub))
target = int(TARGET_HOURS * 3600 / 3)
sub = sub.head(target)
for i, row in enumerate(sub.itertuples()):
    arr = row.audio["array"]; sr = row.audio["sampling_rate"]
    a = librosa.resample(arr.astype("float32"), orig_sr=sr, target_sr=22050)
    if a.ndim > 1: a = a.mean(axis=1)
    p = f"data/cv_{i:05d}.wav"
    sf.write(p, a, 22050)
    open(p.replace(".wav", ".txt"), "w", encoding="utf-8").write(row.sentence)
print("hazır:", len(glob.glob("data/*.wav")))

### (B) Sentetik / Recording — zip yükle
`DATA_SOURCE == synthetic` (veya recording) ise bu hücreyi çalıştır: `synthetic.zip` (wav+txt çiftleri) yükle ve `data/` klasörüne aç.

In [ ]:
# 2.B) Sentetik / Recording verisi yükle
from google.colab import files
import glob, os, zipfile
os.makedirs("data", exist_ok=True)
print("synthetic.zip (wav+txt çiftleri) yükle:")
up = files.upload()
for fn in up:
    if fn.endswith(".zip"):
        with zipfile.ZipFile(fn, "r") as z:
            z.extractall("data")
        print("açıldı:", fn)
print("data/wav:", len(glob.glob("data/*.wav")))

## 3) Piper metadata hazırla + eğit

In [ ]:
# 3) Piper verisini hazırla (wav+txt çiftlerinden)
# train_prepare, data/ içindeki wav+txt çiftlerinden config.json üretir.
!piper.train_prepare --dataset-dir data --sample-rate 22050 --language ckb --config-out config.json
print("config ok")
print("wav:", !ls data/*.wav | wc -l)

In [ ]:
# 4) EĞİT (Colab GPU; süre veriye göre)
!piper.train --config-dir . --run-name sorani-female
print("eğitim bitti")

In [ ]:
# 5) ONNX aktar + indir
!piper.export_onnx --run-dir runs --output-dir output --dataset-dir data
!ls -la output/
from google.colab import files
import glob
for f in sorted(glob.glob("output/*")):
    print("indir:", f)
    files.download(f)